# Notebook 02 — Proposal 2: Participant-Normalized Time Moderation (L3.8)

**CS19 Capstone Project**

## Scope
This notebook integrates the **Proposal 2 participant-normalized time moderation** signal into the
Model C / Proposal 1 L3 usability scores at the **L3.8** step.

### What Proposal 2 does and does not do
- **Does**: Apply a normalized-duration cap to the **L3 usability score only** when a participant's
  task time is substantially longer than the participant-adjusted task median.
- **Does not**: Change L1, L2, Accessibility scores, Proposal 1 caps, or explanation fields.
- **Does not**: Raise any score. Scores can only stay the same or go down.
- **Does not**: Treat retry/confusion language as a direct scoring rule; time is a moderation signal.

### Cap thresholds (normalized_median_ratio)
| ratio | cap applied |
|---|---|
| ≥ 4× | 25 |
| ≥ 3× | 40 |
| ≥ 2× | 50 |
| < 2× | no cap |

### Participant speed-factor normalization
`speed_factor_raw = participant_total_actual / participant_total_recommended`  
`speed_factor = clip(speed_factor_raw, 0.7, 1.5)`  
`normalized_duration = duration_minutes / speed_factor`  
`normalized_median_ratio = normalized_duration / task_normalized_median`

## Outputs
All outputs written to `outputs/` relative to this notebook's directory.

> **Note**: Accessibility scores from Notebook 01 are **not changed** by this notebook.
> Tester Quality (Notebook 04) is an independent third score and is **not multiplied** into these results.

In [1]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd

BASE_DIR    = Path(".").resolve()
INPUT_DIR   = BASE_DIR
OUTPUT_DIR  = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs from Notebook 01
ROWS_PATH     = OUTPUT_DIR / "proposal1_rows.csv"
INSIGHTS_PATH = OUTPUT_DIR / "proposal1_df_insights.csv"

# Raw time signal
TIME_PATH = INPUT_DIR / "proposal2_time_signal_result.csv"

# Validation reference (already uploaded)
PROVIDED_L3_PATH = INPUT_DIR / "l3_score_before_time.csv"

for p in [ROWS_PATH, INSIGHTS_PATH, TIME_PATH]:
    assert p.exists(), f"Missing: {p}"

print("BASE_DIR  :", BASE_DIR)
print("Rows      :", ROWS_PATH.name, "exists:", ROWS_PATH.exists())
print("Insights  :", INSIGHTS_PATH.name, "exists:", INSIGHTS_PATH.exists())
print("Time CSV  :", TIME_PATH.name, "exists:", TIME_PATH.exists())

BASE_DIR  : /home/ubuntu/cs19_capstone
Rows      : proposal1_rows.csv exists: True
Insights  : proposal1_df_insights.csv exists: True
Time CSV  : proposal2_time_signal_result.csv exists: True


In [2]:
# Recommended task durations (minutes) keyed by task_id
RECOMMENDED = {1:10, 2:10, 3:10, 4:10, 5:15, 6:10, 7:15, 8:15, 9:15, 10:10, 11:0.5}

def keyify(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    s = re.sub(r'[\s_\-]+', '', s)
    s = re.sub(r'[^a-z0-9]+', '', s)
    return s

def task_key_from_model_task(task):
    if pd.isna(task): return np.nan
    t = str(task).lower()
    if 'aami website' in t:                                      return 'aami_website'
    if 'pdf' in t or 'product disclosure' in t:                  return 'pdfs_and_understanding_terms'
    if 'equal access' in t or 'critical information' in t or 'emergency' in t:
        return 'equal_access_to_critical_information'
    if 'finding answers' in t:                                   return 'finding_answers'
    return np.nan

def task_key_from_time_task_id(task_id):
    try:
        tid = int(task_id)
    except Exception:
        return np.nan
    return {2:'aami_website', 5:'pdfs_and_understanding_terms',
            7:'equal_access_to_critical_information', 10:'finding_answers'}.get(tid, np.nan)

def proposal2_time_cap(ratio):
    if pd.isna(ratio): return np.nan
    if ratio >= 4:  return 25.0
    if ratio >= 3:  return 40.0
    if ratio >= 2:  return 50.0
    return np.nan

def time_signal_from_ratio(ratio):
    if pd.isna(ratio): return np.nan
    if ratio < 1.2:  return 'normal'
    if ratio < 1.5:  return 'mild_time_friction'
    if ratio < 2.0:  return 'medium_time_friction'
    return 'strong_time_friction'

def apply_proposal2_time_cap(score, cap):
    if pd.isna(cap): return score
    return min(score, cap)

def confidence_rank(x):
    return {'high':3,'medium':2,'low':1}.get(str(x).lower().strip(), 0)

def make_explanation(row):
    cap   = row.get('proposal2_time_cap', np.nan)
    ratio = row.get('normalized_median_ratio', np.nan)
    before = row.get('l3_score_before_time', np.nan)
    after  = row.get('l3_score_after_proposal2_time', np.nan)
    if pd.isna(row.get('task_id', np.nan)):
        return 'No Proposal 2 time cap applied because no matching Proposal 2 time evidence was available for this Model C row.'
    if pd.isna(cap):
        return 'No Proposal 2 time cap applied. Normalized task duration did not exceed the time threshold.'
    msg = (f'Proposal 2 time cap applied. Normalized duration was {ratio:.2f}x the normalized task median, '
           f'so the L3 usability score cannot exceed {int(cap)}.')
    if bool(row.get('proposal2_changed_score', False)):
        msg += f' The L3 usability score changed from {before:.2f} to {after:.2f}.'
    else:
        msg += ' Cap was triggered, but the existing L3 score was already below the cap, so the score did not change.'
    return msg

print("Helper functions defined.")

Helper functions defined.


## 1. Load Model C / Proposal 1 outputs

In [3]:
rows       = pd.read_csv(ROWS_PATH)
df_insights = pd.read_csv(INSIGHTS_PATH)
print('Loaded rows:', rows.shape)
print('Loaded insights:', df_insights.shape)
rows.head(3)

Loaded rows: (136, 18)
Loaded insights: (571, 17)


,email,raw_score,met_raw,task,survey_question,ceiling_score,met_expectations,primary_cohort,acc_weight,usa_weight,project_blocked,l1_score,l2_score,l3_score,acc_score_final,usa_score_final,acc_explanation,usa_explanation
0,reneerussell99@hotmail.com,4,Exceeded my expectations — it was smoother and...,AAMI Website,"Overall, how accessible did you find the AAMI ...",80.0,exceeded,ND,1.0,0.4,False,80.0,80.00,80.0,80.00,80.0,No friction rules changed this participant's A...,No friction rules changed this participant's U...
1,reececraigie@outlook.com,4,Met my expectations,AAMI Website,"Overall, how accessible did you find the AAMI ...",80.0,met,AB,0.4,1.0,False,80.0,74.08,67.0,74.08,67.0,This participant's ACC score of 74 reflects a ...,This participant's USA score of 67 reflects a ...
2,giuliaclemente26@gmail.com,3,Partly met my expectations — some areas worked...,AAMI Website,"Overall, how accessible did you find the AAMI ...",60.0,partly_met,ESL,1.0,0.4,False,60.0,51.00,56.4,51.00,56.4,This participant's ACC score of 51 reflects a ...,This participant's USA score of 56 reflects a ...


## 2. Extract `l3_score_before_time` from Model C rows

In [4]:
model_c_scores = rows.copy()
model_c_scores['l3_score_before_time'] = model_c_scores['l3_score']
model_c_scores['model_c_task']         = model_c_scores['task']
keep_cols = [
    'email','task','model_c_task','survey_question','ceiling_score','l1_score','l2_score',
    'l3_score_before_time','acc_score_final','usa_score_final','acc_explanation','usa_explanation',
    'met_expectations','primary_cohort','acc_weight','usa_weight','project_blocked'
]
model_c_scores = model_c_scores[[c for c in keep_cols if c in model_c_scores.columns]].copy()
model_c_scores['email']            = model_c_scores['email'].str.lower().str.strip()
model_c_scores['model_c_task_key'] = model_c_scores['model_c_task'].apply(task_key_from_model_task)

# Export l3_score_before_time.csv
l3_export = model_c_scores[['email','model_c_task','survey_question',
                             'l3_score_before_time','usa_score_final','usa_explanation']].copy()
l3_export.to_csv(OUTPUT_DIR / 'l3_score_before_time.csv', index=False)
print('model_c_scores shape:', model_c_scores.shape)
print('l3_score_before_time.csv written.')

# Optional: validate against provided reference
if PROVIDED_L3_PATH.exists():
    provided = pd.read_csv(PROVIDED_L3_PATH)
    comp_new = model_c_scores[['email','model_c_task','l3_score_before_time','usa_score_final']].copy()
    comp_old = provided[[c for c in ['email','model_c_task','l3_score_before_time','usa_score_final']
                          if c in provided.columns]].copy()
    for df in [comp_new, comp_old]:
        df['email']            = df['email'].astype(str).str.lower().str.strip()
        df['model_c_task']     = df['model_c_task'].astype(str).str.strip()
        for c in ['l3_score_before_time','usa_score_final']:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors='coerce').round(6)
    comp_new = comp_new.sort_values(list(comp_new.columns)).reset_index(drop=True)
    comp_old = comp_old.sort_values(list(comp_old.columns)).reset_index(drop=True)
    status = 'PASS' if comp_new.equals(comp_old) else 'MISMATCH'
    print(f'l3_score_before_time validation vs provided reference: {status}')

model_c_scores shape: (136, 18)
l3_score_before_time.csv written.
l3_score_before_time validation vs provided reference: PASS


## 3. Recompute Proposal 2 time normalization

The raw `proposal2_time_signal_result.csv` is loaded and normalization fields are recomputed
from scratch to ensure reproducibility:
- `speed_factor_raw` = participant total actual / participant total recommended
- `speed_factor` = clip(speed_factor_raw, 0.7, 1.5)
- `normalized_duration` = duration_minutes / speed_factor
- `normalized_task_median` = median of normalized_duration per task_id
- `normalized_median_ratio` = normalized_duration / normalized_task_median

In [5]:
time_raw = pd.read_csv(TIME_PATH)
pre_clip_min = pd.to_numeric(time_raw.get('speed_factor', pd.Series(dtype=float)), errors='coerce').min()
pre_clip_max = pd.to_numeric(time_raw.get('speed_factor', pd.Series(dtype=float)), errors='coerce').max()
print(f'Original speed_factor range: [{pre_clip_min:.3f}, {pre_clip_max:.3f}]')

time_result = time_raw.copy()
time_result['task_id']              = pd.to_numeric(time_result['task_id'], errors='coerce').astype('Int64')
time_result['duration_minutes']     = pd.to_numeric(time_result['duration_minutes'], errors='coerce')
time_result['recommended_minutes']  = time_result['task_id'].map(RECOMMENDED).astype(float)
time_result = time_result[time_result['recommended_minutes'].notna()].copy()
time_result = time_result[time_result['duration_minutes'] > 0].copy()

time_result['participant_total_actual']      = time_result.groupby('participant')['duration_minutes'].transform('sum')
time_result['participant_total_recommended'] = time_result.groupby('participant')['recommended_minutes'].transform('sum')
time_result['speed_factor_raw']  = time_result['participant_total_actual'] / time_result['participant_total_recommended']
time_result['speed_factor']      = time_result['speed_factor_raw'].clip(0.7, 1.5)
time_result['normalized_duration'] = time_result['duration_minutes'] / time_result['speed_factor']
time_result['normalized_task_median'] = time_result.groupby('task_id')['normalized_duration'].transform('median')
time_result['normalized_median_ratio'] = time_result['normalized_duration'] / time_result['normalized_task_median']
time_result['proposal2_time_cap'] = time_result['normalized_median_ratio'].apply(proposal2_time_cap)
time_result['time_signal']        = time_result['normalized_median_ratio'].apply(time_signal_from_ratio)

post_clip_min = time_result['speed_factor'].min()
post_clip_max = time_result['speed_factor'].max()
print(f'Recomputed clipped speed_factor range: [{post_clip_min:.3f}, {post_clip_max:.3f}]')
print(f'Time rows after filtering: {len(time_result)}')
time_result[['participant','task_id','duration_minutes','speed_factor','normalized_median_ratio','proposal2_time_cap','time_signal']].head(5)

Original speed_factor range: [0.155, 5.038]
Recomputed clipped speed_factor range: [0.700, 1.500]
Time rows after filtering: 138


,participant,task_id,duration_minutes,speed_factor,normalized_median_ratio,proposal2_time_cap,time_signal
0,asma,1,12.300000,1.051429,2.494069,50.0,strong_time_friction
1,asma,2,14.316667,1.051429,2.094830,50.0,strong_time_friction
2,asma,4,15.083333,1.051429,1.581400,NaN,medium_time_friction
3,asma,5,22.283333,1.051429,2.124397,50.0,strong_time_friction
4,asma,6,0.116667,1.051429,0.006391,NaN,normal


## 4. Participant → email mapping

Participant labels in the time file are mapped to tester emails using the df_insights table.
Three methods are tried in order: exact normalized tester-name key, exact email-local key,
unique tester-name containment.

In [6]:
name_email_map = df_insights[['tester_name','tester_email']].dropna().drop_duplicates().copy()
name_email_map['participant_key']  = name_email_map['tester_name'].apply(keyify)
name_email_map['email']            = name_email_map['tester_email'].astype(str).str.lower().str.strip()
name_email_map['email_local_key']  = name_email_map['email'].str.split('@').str[0].apply(keyify)

map_records = []
for _, r in name_email_map.iterrows():
    map_records.append({'candidate_key': r['participant_key'], 'email': r['email'],
                        'tester_name': r['tester_name'], 'mapping_method': 'tester_name_exact'})
    map_records.append({'candidate_key': r['email_local_key'], 'email': r['email'],
                        'tester_name': r['tester_name'], 'mapping_method': 'email_local_exact'})
map_df = pd.DataFrame(map_records).dropna(subset=['candidate_key']).drop_duplicates()
exact_counts = map_df.groupby('candidate_key')['email'].nunique().reset_index(name='n_email')
map_df = map_df.merge(exact_counts, on='candidate_key', how='left')
map_df_unique = map_df[map_df['n_email']==1].drop_duplicates('candidate_key')

time_result['participant_key'] = time_result['participant'].apply(keyify)
time_result = time_result.merge(
    map_df_unique[['candidate_key','email','tester_name','mapping_method']]
        .rename(columns={'candidate_key':'participant_key'}),
    on='participant_key', how='left')

# Containment fallback
unmatched_keys = time_result.loc[time_result['email'].isna(),'participant_key'].dropna().unique().tolist()
for pkey in unmatched_keys:
    if len(str(pkey)) < 3:
        continue
    candidates = name_email_map[
        name_email_map['participant_key'].apply(
            lambda nk: isinstance(nk,str) and (pkey in nk or nk in pkey))]
    if candidates['email'].nunique() == 1:
        row_c = candidates.iloc[0]
        mask = (time_result['participant_key']==pkey) & (time_result['email'].isna())
        time_result.loc[mask,'email']          = row_c['email']
        time_result.loc[mask,'tester_name']    = row_c['tester_name']
        time_result.loc[mask,'mapping_method'] = 'tester_name_unique_containment'

unmatched_participants = time_result[time_result['email'].isna()][
    ['participant','participant_key','task_id','task_title']].drop_duplicates()
unmatched_participants.to_csv(OUTPUT_DIR/'proposal2_unmatched_participants.csv', index=False)
print(f'Unmatched participants: {unmatched_participants["participant"].nunique()}')
print(unmatched_participants['participant'].drop_duplicates().tolist())

time_result['model_c_task_key'] = time_result['task_id'].apply(task_key_from_time_task_id)
print(f'Time rows with task key: {time_result["model_c_task_key"].notna().sum()}')

Unmatched participants: 0
[]
Time rows with task key: 53


## 5. Merge and apply Proposal 2 time cap

Model C rows are the left-hand table. Time evidence is deduplicated at `email + model_c_task_key`
level (prefer higher confidence, then higher ratio). The cap is applied only to `l3_score_before_time`.

**Quality check**: assert no score increased.

In [7]:
time_merge_cols = [
    'email','participant','participant_key','model_c_task_key','task_id','task_title',
    'duration_minutes','recommended_minutes','speed_factor_raw','speed_factor',
    'normalized_duration','normalized_task_median','normalized_median_ratio',
    'proposal2_time_cap','time_signal','confidence','mapping_method'
]
time_for_merge = time_result[time_result['email'].notna() & time_result['model_c_task_key'].notna()].copy()
time_for_merge['_confidence_rank'] = time_for_merge['confidence'].apply(confidence_rank)
duplicate_mask  = time_for_merge.duplicated(['email','model_c_task_key'], keep=False)
duplicate_rows  = int(duplicate_mask.sum())
duplicate_pairs = int(time_for_merge.loc[duplicate_mask,['email','model_c_task_key']].drop_duplicates().shape[0])
time_for_merge  = time_for_merge.sort_values(
    ['email','model_c_task_key','_confidence_rank','normalized_median_ratio','duration_minutes'],
    ascending=[True,True,False,False,False]
).drop_duplicates(['email','model_c_task_key'], keep='first')

merged = model_c_scores.merge(time_for_merge[time_merge_cols], on=['email','model_c_task_key'], how='left')
merged['l3_score_after_proposal2_time'] = merged.apply(
    lambda r: apply_proposal2_time_cap(r['l3_score_before_time'], r['proposal2_time_cap']), axis=1)
merged['proposal2_changed_score'] = (
    pd.to_numeric(merged['l3_score_after_proposal2_time'], errors='coerce').round(10)
    != pd.to_numeric(merged['l3_score_before_time'], errors='coerce').round(10))

# Quality check: scores only go down
violations = merged[
    pd.to_numeric(merged['l3_score_after_proposal2_time'], errors='coerce')
    > pd.to_numeric(merged['l3_score_before_time'], errors='coerce') + 1e-9]
assert violations.empty, f'QUALITY CHECK FAILED: {len(violations)} score(s) increased.'
print('Quality check PASSED: no score increased.')

merged['proposal2_time_explanation'] = merged.apply(make_explanation, axis=1)
print(f'Merged rows: {merged.shape}')
print(f'Rows with cap: {int(merged["proposal2_time_cap"].notna().sum())}')
print(f'Rows changed:  {int(merged["proposal2_changed_score"].sum())}')
merged[['email','model_c_task','l3_score_before_time','proposal2_time_cap',
        'l3_score_after_proposal2_time','proposal2_changed_score','time_signal']].head(5)

Quality check PASSED: no score increased.
Merged rows: (136, 36)
Rows with cap: 14
Rows changed:  12


,email,model_c_task,l3_score_before_time,proposal2_time_cap,l3_score_after_proposal2_time,proposal2_changed_score,time_signal
0,reneerussell99@hotmail.com,AAMI Website,80.0,NaN,80.0,False,medium_time_friction
1,reececraigie@outlook.com,AAMI Website,67.0,NaN,67.0,False,NaN
2,giuliaclemente26@gmail.com,AAMI Website,56.4,NaN,56.4,False,normal
3,mgblackwell2001@gmail.com,AAMI Website,32.0,NaN,32.0,False,normal
4,carlpatrickrobinson@gmail.com,AAMI Website,80.0,NaN,80.0,False,normal


## 6. Export outputs

In [8]:
final_cols = [
    'email','participant','participant_key','model_c_task','model_c_task_key','task_id','task_title',
    'survey_question','ceiling_score','l1_score','l2_score','l3_score_before_time',
    'duration_minutes','recommended_minutes','speed_factor_raw','speed_factor',
    'normalized_duration','normalized_task_median','normalized_median_ratio',
    'proposal2_time_cap','l3_score_after_proposal2_time','proposal2_changed_score',
    'time_signal','confidence','proposal2_time_explanation',
    'acc_score_final','usa_score_final','acc_explanation','usa_explanation',
    'met_expectations','primary_cohort'
]
for c in [c for c in final_cols if c not in merged.columns]:
    merged[c] = np.nan
final = merged[final_cols].copy()
final.to_csv(OUTPUT_DIR / 'proposal2_integrated_l3_time_score.csv', index=False)

audit_cols = [
    'email','participant','model_c_task','model_c_task_key','task_id','task_title',
    'l3_score_before_time','proposal2_time_cap','l3_score_after_proposal2_time',
    'proposal2_changed_score','normalized_duration','normalized_task_median',
    'normalized_median_ratio','time_signal','confidence','proposal2_time_explanation'
]
audit = merged[(merged['proposal2_time_cap'].notna()) | (merged['proposal2_changed_score']==True)].copy()
audit = audit[[c for c in audit_cols if c in audit.columns]]
audit.to_csv(OUTPUT_DIR / 'proposal2_time_audit_log.csv', index=False)

time_result.to_csv(OUTPUT_DIR / 'proposal2_time_signal_result_recomputed_for_integration.csv', index=False)

# Unmatched tasks
model_unmatched = model_c_scores[model_c_scores['model_c_task_key'].isna()][
    ['model_c_task','model_c_task_key']].drop_duplicates().copy()
model_unmatched['source'] = 'model_c'
for c in ['task_id','task_title']: model_unmatched[c] = np.nan
model_unmatched = model_unmatched.rename(columns={'model_c_task':'task'})
time_unmatched = time_result[time_result['model_c_task_key'].isna()][
    ['task_id','task_title','model_c_task_key']].drop_duplicates().copy()
time_unmatched['source'] = 'proposal2_time'; time_unmatched['task'] = np.nan
unmatched_tasks = pd.concat([
    model_unmatched[['source','task','model_c_task_key','task_id','task_title']],
    time_unmatched[['source','task','model_c_task_key','task_id','task_title']]
], ignore_index=True)
unmatched_tasks.to_csv(OUTPUT_DIR / 'proposal2_unmatched_tasks.csv', index=False)

print("Output files written:")
for f in ['proposal2_integrated_l3_time_score.csv','proposal2_time_audit_log.csv',
          'proposal2_time_signal_result_recomputed_for_integration.csv',
          'proposal2_unmatched_participants.csv','proposal2_unmatched_tasks.csv',
          'l3_score_before_time.csv']:
    p = OUTPUT_DIR / f
    print(f"  {f}: {p.stat().st_size:,} bytes")

Output files written:
  proposal2_integrated_l3_time_score.csv: 97,574 bytes
  proposal2_time_audit_log.csv: 5,500 bytes
  proposal2_time_signal_result_recomputed_for_integration.csv: 94,170 bytes
  proposal2_unmatched_participants.csv: 47 bytes
  proposal2_unmatched_tasks.csv: 482 bytes
  l3_score_before_time.csv: 38,516 bytes


## 7. Integration summary

In [9]:
model_unique_keys = model_c_scores[['email','model_c_task_key']].dropna().drop_duplicates()
time_check = time_result.copy().merge(
    model_unique_keys.assign(_model_key_exists=True), on=['email','model_c_task_key'], how='left')

summary = {
    'total_model_c_rows':                int(len(model_c_scores)),
    'total_time_rows':                   int(len(time_result)),
    'matched_rows':                      int(merged['task_id'].notna().sum()),
    'unmatched_model_c_rows':            int(merged['task_id'].isna().sum()),
    'unmatched_time_rows':               int(time_check['_model_key_exists'].isna().sum()),
    'rows_with_time_cap':                int(merged['proposal2_time_cap'].notna().sum()),
    'rows_with_score_changed':           int(merged['proposal2_changed_score'].sum()),
    'average_l3_before_time':            float(pd.to_numeric(merged['l3_score_before_time'], errors='coerce').mean()),
    'average_l3_after_proposal2_time':   float(pd.to_numeric(merged['l3_score_after_proposal2_time'], errors='coerce').mean()),
    'pre_recompute_speed_factor_min':    float(pre_clip_min) if not pd.isna(pre_clip_min) else None,
    'pre_recompute_speed_factor_max':    float(pre_clip_max) if not pd.isna(pre_clip_max) else None,
    'post_clip_speed_factor_min':        float(post_clip_min),
    'post_clip_speed_factor_max':        float(post_clip_max),
    'duplicate_time_rows_before_resolution': duplicate_rows,
    'score_increase_violations':         0,
    'proposal2_only_changes_l3_usability': True,
    'accessibility_scores_unchanged':    True,
}
pd.DataFrame([summary]).to_csv(OUTPUT_DIR / 'proposal2_integration_summary.csv', index=False)

human_summary = (
    f"Proposal 2 was integrated into L3.8 time moderation. "
    f"{summary['total_model_c_rows']} Model C rows processed. "
    f"{summary['matched_rows']} rows matched Proposal 2 time evidence. "
    f"{summary['rows_with_time_cap']} rows triggered normalized time caps. "
    f"{summary['rows_with_score_changed']} rows had changed L3 usability scores. "
    f"No score increased. Accessibility scores are unchanged."
)
(OUTPUT_DIR / 'proposal2_human_readable_summary.txt').write_text(human_summary + '\n')
print(human_summary)
print()
print("Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Proposal 2 was integrated into L3.8 time moderation. 136 Model C rows processed. 95 rows matched Proposal 2 time evidence. 14 rows triggered normalized time caps. 12 rows had changed L3 usability scores. No score increased. Accessibility scores are unchanged.

Summary:
  total_model_c_rows: 136
  total_time_rows: 138
  matched_rows: 95
  unmatched_model_c_rows: 41
  unmatched_time_rows: 93
  rows_with_time_cap: 14
  rows_with_score_changed: 12
  average_l3_before_time: 62.85944117647059
  average_l3_after_proposal2_time: 59.09179411764706
  pre_recompute_speed_factor_min: 0.1555
  pre_recompute_speed_factor_max: 5.038333333333333
  post_clip_speed_factor_min: 0.7
  post_clip_speed_factor_max: 1.5
  duplicate_time_rows_before_resolution: 0
  score_increase_violations: 0
  proposal2_only_changes_l3_usability: True
  accessibility_scores_unchanged: True


## 8. Validation note

> **Manual Review** (Notebook 03) provides a reviewer-estimated validation sample.
> It is **not** official ground truth. In the 18-case sample:
> - Model C before-time has the lowest MAE (13.28)
> - Raw survey score MAE: 13.57
> - Proposal 2 after-time MAE: 19.49
>
> Proposal 2 was clearly helpful for case_001 (ghum / AAMI Website) where real navigation
> friction was present. However, several long-duration cases were successful or verbose,
> making the time signal noisy. **Do not overclaim that Proposal 2 is always better.**